# 🏛️ Data Governance Copilot — Playground Notebook

Use this notebook to test every module interactively.
Run cells one by one or run all at once.

---
**How to use:**
- Each section tests one module
- Green output = working correctly
- Change values and re-run to experiment
- Add your own cells at the bottom

## 0️⃣ Setup — run this first every time

In [ ]:
# Setup — run this cell FIRST every time you open the notebook
import sys
from pathlib import Path

# Add src/ to path so all imports work
src_path = str(Path().absolute() / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print(f'✅ src/ added to path: {src_path}')
print(f'✅ Python version: {sys.version}')
print(f'✅ Working directory: {Path().absolute()}')

---
## 1️⃣ Config — settings.py (Day 2)

In [ ]:
# Test config/settings.py
from config.settings import config, DATA_PRODUCTS

print('=== AppConfig ===')
print(f'  enable_mock  : {config.enable_mock}')
print(f'  log_level    : {config.log_level}')
print(f'  debug        : {config.debug}')
print(f'  max_retries  : {config.max_retries}')

print('\n=== LLM Config ===')
print(f'  provider     : {config.llm.provider}')
print(f'  model        : {config.llm.model}')
print(f'  temperature  : {config.llm.temperature}')

print('\n=== Data Products ===')
for name, info in DATA_PRODUCTS.items():
    print(f'  {name:12} → {info["table"]}')

print('\n✅ Config loaded successfully')

In [ ]:
# Experiment: change a config value and see it reflected
print(f'Before: enable_mock = {config.enable_mock}')
config.enable_mock = False
print(f'After:  enable_mock = {config.enable_mock}')
config.enable_mock = True  # reset back
print(f'Reset:  enable_mock = {config.enable_mock}')

---
## 2️⃣ Logging — logging_utils.py (Day 3)

In [ ]:
# Test core/logging_utils.py
from core.logging_utils import (
    setup_logger, with_retry, safe_execute,
    DataSourceError, TicketingError, AgentError
)

log = setup_logger('notebook.test')

log.info('This is an INFO message')
log.warning('This is a WARNING message')
log.error('This is an ERROR message')

print('\n✅ Logger working — check logs/copilot.log for JSON output')

In [ ]:
# Test @with_retry decorator
attempt_count = 0

@with_retry(max_retries=3, delay_seconds=0.01)
def flaky_function():
    global attempt_count
    attempt_count += 1
    if attempt_count < 3:
        raise ConnectionError(f'Simulated failure on attempt {attempt_count}')
    return f'Success on attempt {attempt_count}'

attempt_count = 0
result = flaky_function()
print(f'Result   : {result}')
print(f'Attempts : {attempt_count}')
print('✅ @with_retry working')

In [ ]:
# Test safe_execute
result_ok   = safe_execute(lambda: 42,       fallback=0)
result_fail = safe_execute(lambda: 1/0,      fallback='safe!')
result_none = safe_execute(lambda: None,     fallback='default')

print(f'Normal execution : {result_ok}')
print(f'Failed execution : {result_fail}')
print(f'None result      : {result_none}')
print('✅ safe_execute working')

In [ ]:
# Test exception hierarchy
try:
    raise DataSourceError(
        'Databricks unreachable',
        agent_name='information_agent',
        recoverable=True
    )
except AgentError as e:  # parent catches child
    print(f'Caught as AgentError : {e}')
    print(f'agent_name           : {e.agent_name}')
    print(f'recoverable          : {e.recoverable}')

print('✅ Exception hierarchy working')

---
## 3️⃣ Base Agent — base_agent.py (Day 4)

In [ ]:
# Test core/base_agent.py
from core.base_agent import BaseAgent, AgentRequest, AgentResult

# Test AgentRequest
req = AgentRequest(
    query         = 'Why did retention drop?',
    data_products = ['retention'],
    time_range    = 'last_month',
)
print('=== AgentRequest ===')
print(f'  query         : {req.query}')
print(f'  query_id      : {req.query_id}')  # auto-generated
print(f'  data_products : {req.data_products}')
print(f'  time_range    : {req.time_range}')
print(f'  intent        : "{req.intent}"')   # empty by default

In [ ]:
# Test that two requests get different query_ids
req1 = AgentRequest(query='query one')
req2 = AgentRequest(query='query two')
print(f'req1 id : {req1.query_id}')
print(f'req2 id : {req2.query_id}')
assert req1.query_id != req2.query_id
print('✅ Unique query IDs generated')

In [ ]:
# Build a minimal concrete agent and test it
class EchoAgent(BaseAgent):
    name = 'echo_agent'
    def _execute(self, request: AgentRequest) -> AgentResult:
        return AgentResult(
            agent_name = self.name,
            success    = True,
            data       = {'echoed': request.query},
            summary    = f'Echo: {request.query}',
            confidence = 1.0,
        )

echo = EchoAgent()
result = echo.execute(req)

print('=== AgentResult ===')
print(f'  agent_name        : {result.agent_name}')
print(f'  success           : {result.success}')
print(f'  summary           : {result.summary}')
print(f'  execution_time_ms : {result.execution_time_ms}')
print(f'  confidence        : {result.confidence}')
print(f'  timestamp         : {result.timestamp}')
print('\n✅ BaseAgent template method working')

In [ ]:
# Test that BaseAgent catches crashes gracefully
class CrashAgent(BaseAgent):
    name = 'crash_agent'
    def _execute(self, request):
        raise RuntimeError('Simulated crash!')

crash = CrashAgent()
crash_result = crash.execute(req)

print(f'success : {crash_result.success}')   # False
print(f'error   : {crash_result.error}')     # Simulated crash!
assert crash_result.success == False
assert 'Simulated crash' in crash_result.error
print('✅ Crashes are caught and returned as AgentResult(success=False)')

---
## 4️⃣ Information Agent — information_agent.py (Day 6)

In [ ]:
# Test agents/information_agent.py
from agents.information_agent import InformationAgent, MOCK_GENERATORS

agent = InformationAgent(enable_mock=True)
print(f'Agent name   : {agent.name}')
print(f'Mock mode    : {agent.enable_mock}')
print(f'Capabilities : {agent.capabilities}')

In [ ]:
# Test retention query
req = AgentRequest(
    query         = 'Why did retention drop last month?',
    data_products = ['retention'],
    time_range    = 'last_month',
)
result = agent.execute(req)

metrics = result.data['metrics']['retention']
print('=== Retention Metrics ===')
for key, val in metrics.items():
    if key not in ('breakdown', 'time_range'):
        print(f'  {key:30} : {val}')

print(f'\nAnomalies : {result.data["anomalies"]}')
print(f'Confidence: {result.confidence}')
print(f'Sources   : {result.sources}')

In [ ]:
# Test product detection
test_queries = [
    'Why did churn increase this quarter?',
    'Show me ARR and bookings trends',
    'What is our CAC payback period?',
    'Show LTV by segment',
    'What is our NRR and LTV/CAC ratio?',
]

print('=== Product Detection ===')
for q in test_queries:
    detected = agent._detect_products(q)
    print(f'  "{q[:45]}..." → {detected}')

In [ ]:
# Test all mock generators directly
print('=== Mock Generators ===')
for product, generator in MOCK_GENERATORS.items():
    metrics = generator('last_month')
    print(f'\n{product.upper()}')
    for k, v in metrics.items():
        if k not in ('breakdown', 'ltv_by_segment', 'time_range'):
            print(f'  {k:35} : {v}')

In [ ]:
# Test anomaly detection manually
print('=== Anomaly Detection ===')

# Force low GRR to trigger anomaly
low_grr_metrics = {'gross_retention_rate': 79.5, 'at_risk_accounts': 42}
anomalies = agent._detect_anomalies('retention', low_grr_metrics)
print(f'Low GRR anomalies: {anomalies}')

# Force high CAC payback
high_cac = {'payback_period_months': 23.5}
anomalies2 = agent._detect_anomalies('cac', high_cac)
print(f'High CAC anomalies: {anomalies2}')

# Normal values — no anomalies
normal = {'gross_retention_rate': 91.0, 'at_risk_accounts': 12}
anomalies3 = agent._detect_anomalies('retention', normal)
print(f'Normal anomalies: {anomalies3}')  # should be []

---
## 5️⃣ Knowledge Agent — knowledge_agent.py (Day 8)

In [ ]:
# Test agents/knowledge_agent.py
from agents.knowledge_agent import KnowledgeAgent, MOCK_KNOWLEDGE_BASE

kagent = KnowledgeAgent(enable_mock=True)
print(f'Agent       : {kagent.name}')
print(f'Mock mode   : {kagent.enable_mock}')
print(f'KB products : {list(MOCK_KNOWLEDGE_BASE.keys())}')

In [ ]:
# Test topic detection
test_queries = [
    'What is GRR and why did churn increase?',
    'Explain our bookings methodology',
    'How is CAC calculated?',
    'What is the LTV/CAC ratio?',
    'General question about data',
]

print('=== Topic Detection ===')
for q in test_queries:
    topics = kagent._detect_topics(q)
    print(f'  "{q[:45]}" → {topics}')

In [ ]:
# Test full execution — retention query
req = AgentRequest(
    query = 'Why did retention drop?',
    data_products = ['retention'],
)
result = kagent.execute(req)

print('=== Knowledge Result ===')
print(f'Success    : {result.success}')
print(f'Topics     : {result.metadata.get("topics_found")}')
print(f'Sources    : {result.sources}')
print(f'Confidence : {result.confidence}')
print(f'\nSummary preview:')
print(result.summary[:400], '...')

In [ ]:
# Browse the mock knowledge base
print('=== Mock Knowledge Base Contents ===')
for product, entry in MOCK_KNOWLEDGE_BASE.items():
    print(f'\n{product.upper()}')
    print(f'  Source   : {entry["source"]}')
    print(f'  Def      : {entry["definition"][:80]}...')
    print(f'  Context  : {entry["business_context"][:80]}...')
    print(f'  Runbook  : {entry["runbook"]}')

---
## 6️⃣ Supervisor — supervisor_agent.py (Day 7 & 8)

In [ ]:
# Test agents/supervisor_agent.py
from agents.supervisor_agent import SupervisorAgent

supervisor = SupervisorAgent(enable_mock=True)
print(f'Supervisor created')
print(f'Agents registered:')
print(f'  - {supervisor.information_agent.name}')
print(f'  - {supervisor.knowledge_agent.name}')

In [ ]:
# Test full pipeline — retention query
response = supervisor.run(
    query      = 'Why did retention drop last month?',
    time_range = 'last_month',
)

print('=== Supervisor Response ===')
print(f'Success      : {response.success}')
print(f'Agents used  : {response.agents_used}')
print(f'Confidence   : {response.confidence}')
print(f'Sources      : {response.sources}')
print(f'Anomalies    : {response.anomalies}')
print(f'\nSummary (first 500 chars):')
print(response.summary[:500])

In [ ]:
# Test different queries through the full pipeline
queries = [
    ('retention + knowledge', 'Why did churn increase last month?'),
    ('bookings',              'Show me our ARR and bookings metrics'),
    ('cac',                   'What is our CAC payback period?'),
    ('multi-product',         'Show LTV and CAC ratio for last quarter'),
]

print('=== Pipeline Smoke Tests ===')
for label, query in queries:
    resp = supervisor.run(query=query)
    print(f'\n[{label}]')
    print(f'  Query       : {query[:50]}')
    print(f'  Success     : {resp.success}')
    print(f'  Agents used : {resp.agents_used}')
    print(f'  Confidence  : {resp.confidence}')
    print(f'  Anomalies   : {len(resp.anomalies)} detected')

In [ ]:
# Test product detection in supervisor
print('=== Supervisor Product Detection ===')
test_qs = [
    'Why did our GRR drop?',
    'Show total bookings this quarter',
    'What is blended CAC?',
    'Show LTV by enterprise segment',
    'Compare NRR with CAC payback',
]
for q in test_qs:
    products = supervisor._detect_products(q)
    print(f'  "{q[:45]}" → {products}')

---
## 7️⃣ End-to-End — full pipeline test

In [ ]:
# Full end-to-end smoke test — simulates what the UI does
print('=== End-to-End Pipeline Test ===')
print('Simulating UI → Supervisor → Agents → Response\n')

test_cases = [
    ('Why did retention drop last month?',    'last_month'),
    ('Show me bookings and ARR metrics',      'last_quarter'),
    ('What is our CAC and LTV/CAC ratio?',    'last_month'),
    ('Explain our churn rate increase',       'last_month'),
]

sup = SupervisorAgent(enable_mock=True)
all_passed = True

for query, time_range in test_cases:
    resp = sup.run(query=query, time_range=time_range)
    ok = resp.success and resp.summary and len(resp.agents_used) >= 1
    status = '✅' if ok else '❌'
    if not ok:
        all_passed = False
    print(f'{status} "{query[:50]}"')
    print(f'   agents={resp.agents_used} confidence={resp.confidence}')

print(f'\n{"✅ All tests passed!" if all_passed else "❌ Some tests failed"}')

---
## 8️⃣ Scratch pad — your own experiments

In [ ]:
# Use this cell to try your own queries
# Change the query and re-run

MY_QUERY = 'Why did retention drop last month?'

sup = SupervisorAgent(enable_mock=True)
resp = sup.run(query=MY_QUERY)

print(f'Query      : {MY_QUERY}')
print(f'Success    : {resp.success}')
print(f'Agents     : {resp.agents_used}')
print(f'Confidence : {resp.confidence}')
print(f'Anomalies  : {resp.anomalies}')
print(f'\n--- Full Summary ---')
print(resp.summary)

In [ ]:
# Add your own experiments below
